In [1]:
import swanlab
from swanlab.integration.transformers import SwanLabCallback


In [2]:
import transformers
import torch
from peft import PeftModel


In [3]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import bitsandbytes as bnb
import torch
import torch.nn as nn
import transformers
from peft import LoraConfig, PeftConfig
from trl import SFTTrainer
from trl import setup_chat_format
from datasets import Dataset
from transformers import (AutoModelForCausalLM, 
                          AutoTokenizer, 
                          BitsAndBytesConfig, 
                          TrainingArguments, 
                          pipeline, 
                          logging,
                        Trainer)
from sklearn.metrics import (accuracy_score, 
                             classification_report, 
                             confusion_matrix)
from sklearn.model_selection import train_test_split

In [4]:
df = pd.read_excel("tmp5.xlsx")

In [5]:
print(df)

           subreddit   Type                                        Title  \
0    EatingDisorders  theme             idk if this is a eating disorder   
1    EatingDisorders  theme  my gf has an eating disorder how can i be a   
2    EatingDisorders  theme           terrified about the holiday season   
3    EatingDisorders  theme         feeling self conscious about my body   
4    EatingDisorders  theme         help learning to not move constantly   
..               ...    ...                                          ...   
954  EatingDisorders  theme                          guilt over rest day   
955  EatingDisorders  theme     is this just grief or an eating disorder   
956  EatingDisorders  theme                    how to gain appetite back   
957  EatingDisorders  theme    how do you talk to a parent about your ed   
958  EatingDisorders  theme        new post am i experiencing disordered   

             Author                                            Content  \
0            

In [6]:
def shorten_column_names(columns):
    """
    Shortens the column names based on predefined mappings.

    Args:
        columns (list of str): List of column names to shorten.

    Returns:
        list of str: List of shortened column names.
    """
    # Define a dictionary of replacements for shortening
    replacements = {
        'Informational Social Support (P=1/A=0)': 'Info',
        'Emotional Social Support (P=1/A=0)': 'Emo',
        'Esteem Social Support (P=1/A=0)': 'Esteem',
        'Network Social Support (P=1/A=0)': 'Net',
        'Tangible Social Support (P=1/A=0)': 'Tang'
    }

    # Replace long names with shortened names
    shortened_columns = [replacements.get(col, col) for col in columns]
    return shortened_columns

# Example usage
columns = [
    'Informational Social Support (P=1/A=0)',
    'Emotional Social Support (P=1/A=0)',
    'Esteem Social Support (P=1/A=0)',
    'Network Social Support (P=1/A=0)',
    'Tangible Social Support (P=1/A=0)'
]

shortened_columns = shorten_column_names(columns)
print(shortened_columns)

['Info', 'Emo', 'Esteem', 'Net', 'Tang']


In [16]:
df

,subreddit,Type,Title,Author,Content,Pubtime,Informaton self-disclosure (P=1/A=0),Thoughts self-disclosure (P=1/A=0),Feelings self-disclosure (P=1/A=0)
0,EatingDisorders,theme,idk if this is a eating disorder,Mimgyu,I feel guilty eating infront of pepole I would...,2023-03-25 21:08:10,1,0,1
1,EatingDisorders,theme,my gf has an eating disorder how can i be a,asmith312,My gf has an eating disorder. She obsesses ove...,2022-06-01 13:35:23,0,0,0
2,EatingDisorders,theme,terrified about the holiday season,EDPostRequests,So I'm not even sure if I have an eating disor...,2021-11-17 10:36:59,1,0,1
3,EatingDisorders,theme,feeling self conscious about my body,EDPostRequests,Hello everyone I have recently left treatment ...,2020-12-19 20:21:00,1,0,0
4,EatingDisorders,theme,help learning to not move constantly,EDPostRequests,\n\n5 years into recovery but some recent lif...,2020-01-30 10:35:47,1,0,1
...,...,...,...,...,...,...,...,...,...
954,EatingDisorders,theme,guilt over rest day,BimboGalx,I've been working out (gym + steps) several ti...,2023-06-12 01:46:50,1,1,1
955,EatingDisorders,theme,is this just grief or an eating disorder,EDPostRequests,TRIGGER WARNING: suicide\n\n\nI lost someone w...,2020-09-29 09:41:29,1,1,1
956,EatingDisorders,theme,how to gain appetite back,UsedWaffle,"I (21F) am in ED recovery myself, but I just r...",2023-01-02 06:36:55,1,1,1
957,EatingDisorders,theme,how do you talk to a parent about your ed,dreweloise,ok so i'm going to have to give a bit of conte...,2022-07-17 21:27:29,1,1,1


In [12]:
def lables_transfer(a,l):
    list_l=[]
    for i in range(len(a)):
        row=a.iloc[i,]
        lable=''
        for k in l:
            if row[k]==1:
                lable=lable+k
                lable=lable+','
            else:
                lable=lable+'No '+k+','
        list_l.append(lable)
    return list_l

In [8]:
df.columns=shorten_column_names(df)
df['lab']=lables_transfer(df,['Info', 'Emo', 'Esteem', 'Net', 'Tang'])

NameError: name 'lables_transfer' is not defined

In [7]:
#df.loc[:,'status'] = df.loc[:,'status'].str.replace('Bi-Polar','Bipolar')
#df = df[(df.status != "Personality disorder") & (df.status != "Stress") & (df.status != "Suicidal")]
#df.columns=shorten_column_names(df.columns)
df['Title'] = df['Title'].fillna('').astype(str)
df['Content'] = df['Content'].fillna('').astype(str)
df['Title_Content'] = 'Title：' + df['Title'] + ' Submission：' + df['Content']


In [8]:
# Shuffle the DataFrame and select only 3000 rows
df = df.sample(frac=1, random_state=45).reset_index(drop=True)

# Split the DataFrame
train_size = 0.6
eval_size = 0.1

# Calculate sizes
train_end = int(train_size * len(df))
eval_end = train_end + int(eval_size * len(df))

# Split the data
X_train = df[:train_end]
X_eval = df[train_end:eval_end]
X_test = df[eval_end:]

# Define the prompt generation functions
def generate_prompt(data_point):
    return f"""
You are now reading some Reddit submissions, please decide whether feeling self-disclosure is involved. Choose 1 if yes. Else, choose 0. 
text: {data_point["Title_Content"]}
label: {data_point["Feelings self-disclosure (P=1/A=0)"]}""".strip()

def generate_test_prompt(data_point):
    return f"""

You are now reading some Reddit submissions, please decide whether feeling self-disclosure is involved. Choose 1 if yes. Else, choose 0. 

text: {data_point["Title_Content"]}
label: {data_point["Feelings self-disclosure (P=1/A=0)"]}""".strip()

# Generate prompts for training and evaluation data
X_train.loc[:,'text'] = X_train.apply(generate_prompt, axis=1)
X_eval.loc[:,'text'] = X_eval.apply(generate_prompt, axis=1)

# Generate test prompts and extract true labels
y_true = X_test.loc[:,'Feelings self-disclosure (P=1/A=0)']
X_test = pd.DataFrame(X_test.apply(generate_test_prompt, axis=1), columns=["text"])

/tmp/ipykernel_1987/1632702122.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_train.loc[:,'text'] = X_train.apply(generate_prompt, axis=1)
/tmp/ipykernel_1987/1632702122.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_eval.loc[:,'text'] = X_eval.apply(generate_prompt, axis=1)


In [9]:
X_test

,text
670,"You are now reading some Reddit submissions, p..."
671,"You are now reading some Reddit submissions, p..."
672,"You are now reading some Reddit submissions, p..."
673,"You are now reading some Reddit submissions, p..."
674,"You are now reading some Reddit submissions, p..."
...,...
954,"You are now reading some Reddit submissions, p..."
955,"You are now reading some Reddit submissions, p..."
956,"You are now reading some Reddit submissions, p..."
957,"You are now reading some Reddit submissions, p..."


In [11]:
X_train['Feelings self-disclosure (P=1/A=0)'].value_counts()

Feelings self-disclosure (P=1/A=0)
1    340
0    235
Name: count, dtype: int64

In [12]:
# Convert to datasets
train_data = Dataset.from_pandas(X_train[["text"]])
eval_data = Dataset.from_pandas(X_eval[["text"]])

In [24]:
train_data['text'][3]

"You are now reading some Reddit submissions, please decide whether information self-disclosure is involved. Choose 1 if yes. Else, choose 0. \ntext: Title：need advice for if i should help someone Submission：My brothers gf is struggling with her eating atm and is counting calories. I notice because I do the same but he's completely oblivious. I'm not able to help her myself because of my own stuff but I'm wondering if I should bring it up to my brother? Would that be really horrible for her though? I know I would hate it but I think she needs support and it might suck initially but it might be good, idk? What do you think\nlabel: 0"

In [13]:
import accelerate

In [14]:
#base_model_name = "meta-llama/Llama-3.1-8B-Instruct"
from transformers import BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype="float16",
)
model = AutoModelForCausalLM.from_pretrained(
    "autodl-tmp",
    device_map="auto",
    torch_dtype="float16",
    quantization_config=bnb_config,
)
#model = PeftModel.from_pretrained(
    #base_model,
    #'llama-3.1-trade_signal',
    #device_map="auto",
    #torch_dtype="float16",
    #quantization_config=bnb_config, 
#)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained('autodl-tmp')

tokenizer.pad_token_id = tokenizer.eos_token_id

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [15]:
from tqdm import tqdm
from transformers import pipeline

def predict(test, model, tokenizer):
    y_pred = []
    categories = ['1', '0']

    # 只初始化一次 pipeline
    pipe = pipeline(
        task="text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=2,
        temperature=0.1
    )
    
    # tqdm 包装 iterrows，显示进度和描述
    for _, row in tqdm(test.iterrows(), total=len(test), desc="Predicting"):
        prompt = row["text"]
        result = pipe(prompt)
        answer = result[0]['generated_text'].split("label:")[-1].strip()
        
        # 判断类别
        for category in categories:
            if category.lower() in answer.lower():
                y_pred.append(category)
                break
        else:
            y_pred.append("none")
    
    return y_pred

y_pred = predict(X_test, model, tokenizer)



Device set to use cuda:0
Predicting: 100%|██████████| 289/289 [00:26<00:00, 10.87it/s]


In [17]:
def evaluate(y_true, y_pred):
    import numpy as np
    from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

    # 如果 y_true/y_pred 是字符串，需要转成 int
    y_true = np.array([int(x) for x in y_true])
    y_pred = np.array([int(x) for x in y_pred])

    # 计算总体准确率
    accuracy = accuracy_score(y_true, y_pred)
    print(f'Overall Accuracy: {accuracy:.3f}')

    # 分类报告
    print('\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=['0', '1'], zero_division=0))

    # 混淆矩阵
    print('\nConfusion Matrix:')
    print(confusion_matrix(y_true, y_pred))

# 调用
evaluate(y_true, y_pred)



Overall Accuracy: 1.000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       114
           1       1.00      1.00      1.00       175

    accuracy                           1.00       289
   macro avg       1.00      1.00      1.00       289
weighted avg       1.00      1.00      1.00       289


Confusion Matrix:
[[114   0]
 [  0 175]]


In [40]:
print(y_pred)

['0', '1', '0', '0', '0', '1', '0', '0', '1', '1', '0', '1', '1', '1', '0', '0', '0', '0', '1', '1', '0', '0', '1', '1', '1', '0', '0', '1', '0', '1', '1', '1', '1', '1', '0', '0', '0', '1', '1', '1', '1', '1', '1', '1', '1', '1', '0', '1', '0', '1', '1', '1', '1', '1', '1', '1', '0', '1', '0', '1', '0', '1', '0', '1', '1', '1', '0', '0', '1', '0', '1', '1', '0', '0', '0', '1', '0', '0', '0', '0', '0', '0', '1', '1', '0', '1', '0', '0', '0', '1', '1', '1', '1', '0', '0', '0', '0', '1', '1', '1', '1', '1', '0', '1', '0', '0', '1', '0', '1', '1', '0', '1', '1', '1', '0', '1', '0', '1', '0', '0', '1', '1', '0', '0', '0', '0', '0', '0', '0', '1', '0', '1', '0', '0', '0', '1', '0', '0', '0', '0', '1', '1', '1', '1', '0', '0', '1', '1', '0', '0', '0', '1', '1', '0', '1', '0', '1', '1', '0', '0', '0', '0', '1', '1', '1', '0', '0', '1', '1', '0', '0', '0', '0', '0', '0', '1', '0', '0', '1', '1', '0', '1', '0', '1', '1', '0', '0', '1', '0', '0', '1', '0', '0', '0', '1', '0', '0', '0', '0', '1',

In [18]:
import bitsandbytes as bnb

def find_all_linear_names(model):
    cls = bnb.nn.Linear4bit
    lora_module_names = set()
    for name, module in model.named_modules():
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])
    if 'lm_head' in lora_module_names:  # needed for 16 bit
        lora_module_names.remove('lm_head')
    return list(lora_module_names)
modules = find_all_linear_names(model)
modules

['v_proj', 'gate_proj', 'up_proj', 'k_proj', 'down_proj', 'o_proj', 'q_proj']

swanlab: Tracking run with swanlab version 0.6.3                                   
swanlab: Run data will be saved locally in /root/swanlog/run-20250620_163251-c6a7ee39
swanlab: 👋 Hi IrisCHENHuahua04, welcome to swanlab!
swanlab: Syncing run swan-1 to the cloud
swanlab: 🏠 View project at https://swanlab.cn/@IrisCHENHuahua04/douban_yanshizheng
swanlab: 🚀 View run at https://swanlab.cn/@IrisCHENHuahua04/douban_yanshizheng/runs/nr16rb3s0w2i0rtdd0wdg


In [19]:
output_dir="llama-3.1-douban_chinese_yanshizheng"

peft_config = LoraConfig(
    lora_alpha=16,
    lora_dropout=0,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=modules,
)

training_arguments = TrainingArguments(
    output_dir=output_dir,                    # directory to save and repository id
    num_train_epochs=1,                       # number of training epochs
    per_device_train_batch_size=1,            # batch size per device during training
    gradient_accumulation_steps=8,            # number of steps before performing a backward/update pass
    gradient_checkpointing=True,              # use gradient checkpointing to save memory
    optim="paged_adamw_32bit",
    logging_steps=1,                         
    learning_rate=2e-4,                       # learning rate, based on QLoRA paper
    weight_decay=0.001,
    fp16=True,
    bf16=False,
    max_grad_norm=0.3,                        # max gradient norm based on QLoRA paper
    max_steps=-1,
    warmup_ratio=0.03,                        # warmup ratio based on QLoRA paper
    group_by_length=False,
    lr_scheduler_type="cosine",               # use cosine learning rate scheduler
    #report_to="wandb",                  # report metrics to w&b
    report_to="none",
    eval_strategy="steps",              # save checkpoint every epoch
    eval_steps = 0.2,
     #max_length=512
    
)
swanlab_callback = SwanLabCallback(
    project="huggingface", 
    experiment_name="LLama3.1ts3"
)
trainer = SFTTrainer(
    model=model,
    args=training_arguments,
    train_dataset=train_data,
    eval_dataset=eval_data,
    peft_config=peft_config,
    callbacks=[swanlab_callback],
    #dataset_text_field='text',
    #tokenizer=tokenizer,
    
    #packing=False,
    #dataset_kwargs={
    #"add_special_tokens": False,
    #"append_concat_token": False,
    #}
)

Converting train dataset to ChatML:   0%|          | 0/575 [00:00<?, ? examples/s]

Applying chat template to train dataset:   0%|          | 0/575 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/575 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/575 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/95 [00:00<?, ? examples/s]

Applying chat template to eval dataset:   0%|          | 0/95 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/95 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/95 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [20]:
swanlab.init(
    # 设置项目名
    project="douban_yanshizheng"
)
tokenizer.pad_token = tokenizer.eos_token
trainer.train()

swanlab: swanlab version 0.6.4 is available!  Upgrade: `pip install -U swanlab`    
swanlab: Tracking run with swanlab version 0.6.3                                   
swanlab: Run data will be saved locally in /root/swanlog/run-20250625_093007-a3b1799d
swanlab: 👋 Hi IrisCHENHuahua04, welcome to swanlab!
swanlab: Syncing run penguin-16 to the cloud
swanlab: 🏠 View project at https://swanlab.cn/@IrisCHENHuahua04/douban_yanshizheng
swanlab: 🚀 View run at https://swanlab.cn/@IrisCHENHuahua04/douban_yanshizheng/runs/6mgie8vqf94cu4rew2ie0


Step,Training Loss,Validation Loss
15,2.657700,2.205599
30,2.294900,2.138509
45,2.067700,2.114645
60,2.029500,2.106349


TrainOutput(global_step=71, training_loss=2.2386807623043867, metrics={'train_runtime': 230.9993, 'train_samples_per_second': 2.489, 'train_steps_per_second': 0.307, 'total_flos': 7066691890814976.0, 'train_loss': 2.2386807623043867})

In [34]:
y_pred = predict(X_test, model, tokenizer)


Device set to use cuda:0
Predicting:   0%|          | 0/289 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/root/miniconda3/lib/python3.10/site-packages/torch/utils/checkpoint.py:61: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
Predicting: 100%|██████████| 289/289 [00:43<00:00,  6.61it/s]


In [37]:
#swanlab.finish()
evaluate(y_true, y_pred)


Overall Accuracy: 1.000

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       108
           1       1.00      1.00      1.00       181

    accuracy                           1.00       289
   macro avg       1.00      1.00      1.00       289
weighted avg       1.00      1.00      1.00       289


Confusion Matrix:
[[108   0]
 [  0 181]]


In [56]:
for i in range(len(y_pred)):
    if y_pred[i]=='none':
        y_pred[i]='no'


swanlab: 🏠 View project at https://swanlab.cn/@IrisCHENHuahua04/douban
swanlab: 🚀 View run at https://swanlab.cn/@IrisCHENHuahua04/douban/runs/fkjzdcv8d7vn59wpnoerw
                                                                                                    


In [44]:
# 打印可训练参数
def print_trainable_params(model: torch.nn.Module) -> None:
    trainable_params, all_param = 0, 0
    for param in model.parameters():
        num_params = param.numel()
        # if using DS Zero 3 and the weights are initialized empty
        if num_params == 0 and hasattr(param, "ds_numel"):
            num_params = param.ds_numel
        all_param += num_params
        if param.requires_grad:
            trainable_params += num_params
    print("trainable params: {:d} || all params: {:d} || trainable%: {:.4f}".format(
        trainable_params, all_param, 100 * trainable_params / all_param))

In [55]:
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, TextIteratorStreamer, GenerationConfig
from peft import PeftModel
 
    ## 载入预训练模型
tokenizer = AutoTokenizer.from_pretrained('autodl-tmp', use_fast=True, padding_side="left")
print("Tokenizer Load Success!")
config = AutoConfig.from_pretrained('autodl-tmp')
    # Load and prepare pretrained models (without valuehead).
model = AutoModelForCausalLM.from_pretrained(
        'autodl-tmp',
        config=config,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        revision='main'
    )
print('origin config =', model.config)
    # 模型合并
ckpt_list = ["checkpoint-73"]
for checkpoint in ckpt_list:
    print('Merge checkpoint: {}'.format(checkpoint))
    model = PeftModel.from_pretrained(model, os.path.join('llama-3.1-trade_signal', checkpoint))
    model = model.merge_and_unload()
print('merge config =', model.config)

Tokenizer Load Success!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

origin config = LlamaConfig {
  "architectures": [
    "LlamaForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 128000,
  "eos_token_id": [
    128001,
    128008,
    128009
  ],
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 4096,
  "initializer_range": 0.02,
  "intermediate_size": 14336,
  "max_position_embeddings": 131072,
  "mlp_bias": false,
  "model_type": "llama",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 8,
  "pretraining_tp": 1,
  "rms_norm_eps": 1e-05,
  "rope_scaling": {
    "factor": 8.0,
    "high_freq_factor": 4.0,
    "low_freq_factor": 1.0,
    "original_max_position_embeddings": 8192,
    "rope_type": "llama3"
  },
  "rope_theta": 500000.0,
  "tie_word_embeddings": false,
  "torch_dtype": "float16",
  "transformers_version": "4.52.4",
  "use_cache": true,
  "vocab_size": 128256
}

Merge checkpoint: checkpoint-73
merge config = LlamaConfig {
  "architectures": [
    "LlamaForCau

In [58]:
def apply_lora(model_name_or_path, output_path, lora_path):
    print(f"Loading the base model from {model_name_or_path}")
    base = AutoModelForCausalLM.from_pretrained(
        model_name_or_path, torch_dtype=torch.float16, low_cpu_mem_usage=True
    )
    base_tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
 
    print(f"Loading the LoRA adapter from {lora_path}")
 
    lora_model = PeftModel.from_pretrained(
        base,
        lora_path,
        torch_dtype=torch.float16,
    )
 
    print("Applying the LoRA")
    model = lora_model.merge_and_unload()
 
    print(f"Saving the target model to {output_path}")
    model.save_pretrained(output_path)
    base_tokenizer.save_pretrained(output_path)

In [59]:
apply_lora('autodl-tmp','llama-3.1-trade_signal','llama-3.1-trade_signal/checkpoint-73')

Loading the base model from autodl-tmp


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading the LoRA adapter from llama-3.1-trade_signal/checkpoint-73
Applying the LoRA
Saving the target model to llama-3.1-trade_signal


In [21]:
df2=pd.read_excel('tmp4.xlsx')
df2['Title_Content'] = 'Title：' + df2['Title'].astype(str) + ' Submission：' + df2['Content'].astype(str)


In [22]:
print(df2.head())
print(df2.columns)
#df2['titille+content']=df2['Title']+df2['content ']

   Unnamed: 0.1  Unnamed: 0        subreddit   Type   Title  \
0             0           2  EatingDisorders  theme  advice   
1             1           7  EatingDisorders  theme  advice   
2             2           8  EatingDisorders  theme  advice   
3             3           9  EatingDisorders  theme  advice   
4             4          17  EatingDisorders  theme  advice   

              Author                                            Content  \
0     EDPostRequests  My girlfriend has recently started relapsing i...   
1     EDPostRequests  for the past month or so i'e been struggling w...   
2     EDPostRequests  I don't know what eating disorder I have I res...   
3      Little-red099  I think my coworker/friend is struggling with ...   
4  MoreLeatherPlease  Hi. I am in recovery after a long battle with ...   

              Pubtime  Informaton self-disclosure (P=1/A=0)  \
0 2020-05-03 23:58:25                                     0   
1 2020-08-14 10:15:51                       

In [23]:
# Generate prompts for training and evaluation data
df2['Feelings self-disclosure (P=1/A=0)']=''
df2.loc[:,'text'] = df2.apply(generate_test_prompt, axis=1)
df2
df2_test=df2.loc[:,'text'] 

In [24]:
df2_test=pd.DataFrame(df2_test)
df2_test

,text
0,"You are now reading some Reddit submissions, p..."
1,"You are now reading some Reddit submissions, p..."
2,"You are now reading some Reddit submissions, p..."
3,"You are now reading some Reddit submissions, p..."
4,"You are now reading some Reddit submissions, p..."
...,...
1882,"You are now reading some Reddit submissions, p..."
1883,"You are now reading some Reddit submissions, p..."
1884,"You are now reading some Reddit submissions, p..."
1885,"You are now reading some Reddit submissions, p..."


In [25]:
print(df2_test.iloc[1,:].tolist())

["You are now reading some Reddit submissions, please decide whether feeling self-disclosure is involved. Choose 1 if yes. Else, choose 0. \n\ntext: Title：advice Submission：for the past month or so i'e been struggling with disordered eating. for many reasons i haven't been able to open up to my family, nor my friends who live hours away. because of this, i have felt so alone. \n\ndoes anybody know of any support groups for those who are going through the same thing? preferably free ones?\nlabel:"]


In [26]:
df2_pred = predict(df2_test , model, tokenizer)
#df2_pred

Device set to use cuda:0
Predicting:   0%|          | 0/1887 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/root/miniconda3/lib/python3.10/site-packages/torch/utils/checkpoint.py:61: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
Predicting: 100%|██████████| 1887/1887 [04:10<00:00,  7.53it/s]


In [28]:
df2['Feelings self-disclosure (P=1/A=0)']=df2_pred
#del df2['Title_Content'] 
df2['Feelings self-disclosure (P=1/A=0)'].value_counts()
#df2.to_excel('tmp4.xlsx')

Feelings self-disclosure (P=1/A=0)
1       1161
0        725
none       1
Name: count, dtype: int64

In [31]:
df2.sample(400).to_excel('tmp6.xlsx')

In [46]:
df2_1=df2[df2['Social']=='0']
#df2_1=df2_1[df2_1['Type']=='REPLY']
#del df2_1['text'] 
df2_1=df2_1[~df2_1['Content'].isin(df['Content'])]
#df2_1.to_excel('db_all_reply_social.xlsx')

In [47]:
df3=pd.read_excel('tmp3.xlsx')
df3[df3['Content'].isin(df2_1['Content'])]['Informaton self-disclosure (P=1/A=0)'].value_counts()


Informaton self-disclosure (P=1/A=0)
1.0    176
0.0     92
Name: count, dtype: int64

In [52]:
df2_1[~df2_1['Content'].isin(df3['Content'])].to_excel('tmp4.xlsx')


In [ ]:
df2.to_csv('f1_all_reply pred result.csv',index=False,encoding='utf_8_sig',mode='w')

In [41]:
model_dir = "/root/llama3.1"
model.save_pretrained(model_dir)
tokenizer.save_pretrained(model_dir)

('/root/llama3.1/tokenizer_config.json',
 '/root/llama3.1/special_tokens_map.json',
 '/root/llama3.1/chat_template.jinja',
 '/root/llama3.1/tokenizer.json')

In [89]:
d3=pd.read_csv('d2 pred result.csv')
print(d3.head())

  group name              Title   Type user id   \
0        生活组  饮食障碍 多次🐦朋友约饭 怎么道歉  THEME     momo   
1        生活组  饮食障碍 多次🐦朋友约饭 怎么道歉  REPLY  砂锅粥炖鸡甜汤   
2        生活组  饮食障碍 多次🐦朋友约饭 怎么道歉  REPLY      吸锦鲤   
3        生活组  饮食障碍 多次🐦朋友约饭 怎么道歉  REPLY  覆盆子2025   
4        生活组  饮食障碍 多次🐦朋友约饭 怎么道歉  REPLY   薯条的揪尼仔   

                                            content  number of likes  \
0  我有严重的饮食障碍 焦虑会突然间吃很多然后水肿 衣服穿不下 沮丧 没办法出门因为这个 很不可...               赞   
1     那就不要约啊 说句不好听 关系没到那份上 你又自己清楚自己的病是什么情况 为什么还要答应吃饭         赞 (209)   
2                                            那就一律不要约         赞 (106)   
3                          那就不要答应别人的任何约饭局，答应了🐦会很让人恼火          赞 (87)   
4                 那不能说现在不确定，周五再告诉朋友周末能不能约不行么？总比你直接鸽强          赞 (57)   

                   pubtime                                    titille+content  \
0  2024-04-28 14:09:29 新加坡  饮食障碍 多次🐦朋友约饭 怎么道歉我有严重的饮食障碍 焦虑会突然间吃很多然后水肿 衣服穿不下...   
1   2024-04-28 14:36:23 广东  饮食障碍 多次🐦朋友约饭 怎么道歉那就不要约啊 说句不好听 关系没到那份上 你又自己清楚自己...   
2   2024-04-28 14

In [90]:
del d3['titille+content']

In [91]:
del d3['text']

In [92]:
d3 = d3.rename(columns={'signals': 'ed or not'})

In [106]:
d3.loc[(d3['Title'].str.contains('')) & (d3['Type'] == 'THEME'), 'ed or not'] = 'yes'

In [98]:
d3.loc[(d3['Title'].str.contains('障碍')) & (d3['Type'] == 'theme'), 'ed or not']

Series([], Name: ed or not, dtype: object)

In [107]:
d3['ed or not'].value_counts()

ed or not
yes     32254
no      26326
none      894
Name: count, dtype: int64

In [108]:
d3_yes=d3[d3['ed or not']=='yes']
d3_yes['Title'].value_counts()

Title
（更饮食）因为谷爱凌我动了，半个月6斤                 1284
一人食｜干净饮食真的能瘦！！！                     1089
生活技能｜通过调整饮食和运动从110瘦到了93//更新了运动课表     868
课间聊天｜我已经照莱万的“健康饮食”理念吃了两个多月了          862
英子健身不？控制饮食不？医美走起不？                   793
                                    ... 
讨论/科普｜【讨论】猫猫有进食障碍怎么办？                  1
有点难理解进食障碍这个剧情                          1
问｜有进食障碍经历的姐妹进！跟男朋友在一起后进食障碍逐渐加重         1
8年进食障碍自愈分享                             1
【寻找受访对象】校园学生媒体 想做ED（进食障碍）的题            1
Name: count, Length: 351, dtype: int64

In [102]:
d3_n=d3[d3['ed or not']=='no']

print(d3_n['Title'].value_counts())

Title
日常饮食生活｜按照网上方法做猫饭，猫一口都不吃😭          1682
日常饮食生活｜我觉得有舔毛习惯的小猫不需要洗澡            826
神奇经历｜拥有健康的生活规律饮食起居，是我自己身体的主人       800
大家听过戒麸质饮食的观点吗？                     686
日常饮食生活｜过年小猫独自留守十一天，回家发现小猫胖了一圈      678
                                  ... 
（内含投票）想知道大家都经历过进食障碍吗（厌/暴食）           1
思考及感悟｜学习最先要克服的就是心理障碍                 1
选择性进食障碍（自己编的），和不熟的人一起吃饭根本吃不进去        1
（转）推荐一本自救书籍《战胜暴食的CBT-E方法》            1
卷心菜园｜这个up主分析得真好，我从没想到蒋静的衣服还有隐喻       1
Name: count, Length: 272, dtype: int64


In [105]:
print(d3_n['Title'].unique())


['饮食障碍 多次🐦朋友约饭 怎么道歉' '心得＆分享｜饮食障碍相关博主' '无偿互助｜英文访谈— 一项跨文化的饮食障碍研究'
 '记录｜对抗饮食障碍 好好吃饭' '心得＆分享｜意识到自己的进食障碍“认知饮食限制”'
 '交流解惑｜因为低卡饮食减肥得了进食障碍去了精神中心住院了' '讨论/科普｜【讨论】猫有饮食障碍这个问题，能彻底解决吗？'
 '常识＆科普｜如何运动能活蹦乱跳到一百岁（Outlive《超越百岁》建议整理）+饮食和睡眠'
 '经验分享｜从精力低到活力满分：控糖饮食法真实体验分享' '种草排雷｜饮食思维障碍...'
 '实用小贴士｜解决精神内耗/心境障碍的通用实操办法和成功经验' '生活教程｜写一点减脂期饮食分享。。。。。' 'uu们 你们有饮食障碍吗'
 '经验分享｜最容易做到的精力提升法：在饮食上节省80%的精力' '社会规律｜我发现人类在进化中曾经是活下来的基因本能，在现代反而成了障碍'
 '一人食｜干净饮食真的能瘦！！！' '“能为他们在夺取冠军的路上扫除那么一点点障碍，我是非常非常幸福的！”'
 '不让饮食习惯被资本侵蚀的一些小tips' '有没有哥哥觉得凝血障碍做0才是最带感的。。。最考验1技术的一集。。。。'
 '分享｜轻松又科学的减肥饮食窍门：无需改变饮食和节食' '打卡＆吐槽｜停止在进食障碍开始之前'
 '经验分享｜告别拖延和负面学习心理，教会教会怎么攻克学习障碍，怎么高效学习！（今晚在评论区答疑）' '自然规律｜我发现减肥的尽头是「正常」饮食'
 '个人经验：用进食障碍治疗的all in法则养护自我' '女性燎原日常｜分享一篇文章-中国女性领导干部的晋升障碍与发展路径'
 '想健康减肥的朋友看看“卫健委”出的攻略，饮食方面还贴心的分好了地区。' '治痘抗衰的抗炎饮食到底应该怎么吃？'
 '明天混双不会打成女单障碍赛吧。。' '心得分享｜一份简单实用又省钱的饮食方案' '发送｜公交遇到智力障碍妇女 果断报警'
 '关于创伤/依恋/障碍的meme'
 "阅读障碍者有福了。。Let's 阅读📖。。安利一些。。。可读性比较强的。。。非网络小说文学。。。（偏严肃文学。。。但是好看的。。。）"
 '关于饮食的极简' '社会规律｜我们的正常饮食被各种快餐网红食物替代了' '来看点。。。真实的高功能反社会人格障碍。。。'
 '轻轻复盘男双传奇和男单g

In [109]:
d3_yes.to_csv('豆瓣中文按关联性收集.csv',index=False,encoding='utf_8_sig',mode='w')